In [1]:
from google.colab import files

uploaded = files.upload()

Saving customers.csv to customers.csv
Saving orders.csv to orders.csv


In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, count

spark = SparkSession.builder \
    .appName("OrderAnalysis") \
    .getOrCreate()

customers_df = spark.read.csv(
    "customers.csv",
    header=True,
    inferSchema=True
)

orders_df = spark.read.csv(
    "orders.csv",
    header=True,
    inferSchema=True
)

customers_df.show()
orders_df.show()

joined_df = orders_df.join(
    customers_df,
    on="customer_id",
    how="inner"
)

print("Joined Data")
joined_df.show()

joined_df = joined_df.withColumn(
    "delayed",
    when(col("delay_days") > 0, 1).otherwise(0)
)

delay_by_region = joined_df.groupBy("region") \
    .agg(
        count(
            when(col("delayed") == 1, True)
        ).alias("delayed_orders")
    )

print("Delayed Orders By Region")
delay_by_region.show()

+-----------+-------------+------+
|customer_id|customer_name|region|
+-----------+-------------+------+
|          1|        Rahul| South|
|          2|        Priya| North|
|          3|         Arun| South|
|          4|         John|  East|
+-----------+-------------+------+

+--------+-----------+------------+----------+
|order_id|customer_id|product_name|delay_days|
+--------+-----------+------------+----------+
|     101|          1|      Laptop|         2|
|     102|          2|      Mobile|         0|
|     103|          1|    Keyboard|         5|
|     104|          3|       Mouse|         3|
|     105|          4|     Monitor|         0|
|     106|          2|  Headphones|         4|
+--------+-----------+------------+----------+

Joined Data
+-----------+--------+------------+----------+-------------+------+
|customer_id|order_id|product_name|delay_days|customer_name|region|
+-----------+--------+------------+----------+-------------+------+
|          1|     101|      Lapt

In [3]:
delay_by_region.write \
    .mode("overwrite") \
    .parquet("output/delayed_orders_by_region")

In [7]:
!zip -r delayed_orders_by_region.zip output/delayed_orders_by_region

  adding: output/delayed_orders_by_region/ (stored 0%)
  adding: output/delayed_orders_by_region/._SUCCESS.crc (stored 0%)
  adding: output/delayed_orders_by_region/.part-00000-36b7b0e4-0bf2-40db-8959-a34d78a716b6-c000.snappy.parquet.crc (stored 0%)
  adding: output/delayed_orders_by_region/part-00000-36b7b0e4-0bf2-40db-8959-a34d78a716b6-c000.snappy.parquet (deflated 35%)
  adding: output/delayed_orders_by_region/_SUCCESS (stored 0%)


In [8]:
files.download("delayed_orders_by_region.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [6]:
spark.stop()